In [0]:
from pyspark.sql.functions import col, lit, concat, lower, regexp_replace, split, when, current_timestamp, row_number, coalesce, to_timestamp, last, xxhash64
from pyspark.sql.window import Window
from pyspark.sql.types import LongType
import time

# --------------------------------------------------
# 1. Performance, Widget Setup & Retry Decorator
# --------------------------------------------------
spark.conf.set("spark.sql.shuffle.partitions", 4)
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

dbutils.widgets.text("customer_id", "cust_001")
dbutils.widgets.text("source_system", "salesforce") 
dbutils.widgets.text("object_name", "account")      
dbutils.widgets.text("load_type", "historical")

customer_id = dbutils.widgets.get("customer_id").lower()
source_system = dbutils.widgets.get("source_system").lower()
object_name = dbutils.widgets.get("object_name").lower()
load_type = dbutils.widgets.get("load_type").lower()

def retry_on_failure(retries=3, delay=15):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt} failed for {object_name}: {str(e)}")
                    if attempt == retries: raise e
                    time.sleep(delay)
        return wrapper
    return decorator

# Define Paths 
bucket_path = "s3://hgi-databricks-data-lakehouse-dev/"
landing_zone_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/{load_type}/"
checkpoint_path = f"{bucket_path}layers/bronze/checkpoints/{load_type}/{source_system}/{customer_id}/{object_name}/"

# Define Target Table 
if source_system == "bigquery":
    target_table_name = "crm_events"
else:
    target_table_mapping = {
        "account": "crm_accounts", 
        "contact": "crm_contacts",
        "opportunity": "crm_opportunities", 
        "campaign": "crm_campaigns", 
        "user": "crm_users",
        "event": "crm_events", 
        "task": "crm_tasks",
        "campaignmember": "crm_campaign_members"
    }
    target_table_name = target_table_mapping.get(object_name, f"crm_{object_name}s")

catalog_name = "bronze"
schema_name = customer_id 
table_full_name = f"{catalog_name}.{schema_name}.{target_table_name}"

print(f"🚀 Starting Multi-Tenant Bronze Auto Loader: {source_system} -> {object_name} -> {table_full_name} ({load_type})")

# --------------------------------------------------
# 2. Minimal Target Table Initialization
# --------------------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_full_name} (
        tenant BIGINT,
        id STRING,
        source_system STRING,
        source_system_object STRING,
        source_key_name STRING,
        source_key_value STRING,
        data_timestamp TIMESTAMP,
        created_date TIMESTAMP,
        created_at TIMESTAMP,
        updated_at TIMESTAMP,
        status STRING,
        record_hash STRING
    )
    USING DELTA
    CLUSTER BY (source_system, source_system_object, source_key_name, source_key_value)
""")

# --------------------------------------------------
# 3. Dynamic Normalization & Deduplication Logic
# --------------------------------------------------
def process_bronze_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return

    active_spark = batch_df.sparkSession

    print(f"Processing Batch {batch_id} with {batch_df.count()} raw records...")
    
    # Exclude 'extraction_timestamp' from raw_cols so it doesn't trigger dynamic prefixing
    raw_cols = [c for c in batch_df.columns if c != "extraction_timestamp"]
    sf_object_name = object_name.capitalize()
    
    df_norm = batch_df.withColumn("status", lit("processing"))
    
    # --- A. Client Spec: Schema Normalization & Dynamic Prefixing ---
    if source_system == "salesforce":
        df_norm = df_norm \
            .withColumn("tenant", lit(int(customer_id.split('_')[1])).cast(LongType())) \
            .withColumn("source_system", lit("salesforce")) \
            .withColumn("source_system_object", lit(sf_object_name)) \
            .withColumn("source_key_name", lit("Id")) \
            .withColumn("source_key_value", col("Id")) \
            .withColumn("id", concat(lit(f"salesforce_{sf_object_name}_Id_"), col("Id"))) \
            .withColumn("data_timestamp", coalesce(
                to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),  
                to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ssX"),
                col("SystemModstamp").cast("timestamp") 
            )) \
            .withColumn("systemmodstamp", coalesce(
                to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),  
                to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ssX"),
                col("SystemModstamp").cast("timestamp") 
            )) \
            .withColumn("created_date", coalesce(
                to_timestamp(col("CreatedDate"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
                to_timestamp(col("CreatedDate"), "yyyy-MM-dd'T'HH:mm:ssX"),
                col("CreatedDate").cast("timestamp")
            )) \
            .withColumn("created_at", current_timestamp()) \
            .withColumn("updated_at", current_timestamp())

        # Base known system keys to exclude from dynamic a_ prefixing
        known_system_keys = ['id', 'systemmodstamp', 'createddate', 'extraction_timestamp']

        # --- Object Specific Extraction Logic ---
        if object_name == "account":
            known_system_keys.append('name') 
            df_norm = df_norm.withColumn("name", col("Name")) \
                             .withColumn("domain", when(col("Website").isNotNull(), lower(regexp_replace(regexp_replace(regexp_replace(col("Website"), r'^https?://', ''), r'^www\.', ''), r'/.*$', ''))).otherwise(col("Website")))
        
        elif object_name == "contact":
            known_system_keys.append('email') 
            df_norm = df_norm.withColumn("email", lower(col("Email"))) \
                             .withColumn("domain", when(col("Email").contains("@"), split(lower(col("Email")), "@").getItem(1)).otherwise(lit(None)))
        
        elif object_name in ["task", "event"]:
            known_system_keys.extend(['whoid', 'whatid', 'activitydate'])
            df_norm = df_norm.withColumn("contact_source_system_object",
                when(col("WhoId").startswith("003"), lit("Contact"))
                .when(col("WhoId").startswith("00Q"), lit("Lead"))
                .when(col("WhatId").startswith("001"), lit("Account"))
                .when(col("WhatId").startswith("006"), lit("Opportunity"))
                .when(col("WhatId").startswith("500"), lit("Case"))
                .when(col("WhatId").startswith("701"), lit("Campaign"))
                .otherwise(lit("Unknown"))
            ).withColumn("contact_source_key_value", coalesce(col("WhoId"), col("WhatId")))
            
            if "ActivityDate" in raw_cols:
                df_norm = df_norm.withColumn("event_timestamp", col("ActivityDate").cast("timestamp"))

        elif object_name == "campaignmember":
            known_system_keys.extend(['campaignid', 'leadid', 'contactid'])
            df_norm = df_norm.withColumn("campaign_source_key_value", col("CampaignId")) \
                             .withColumn("contact_source_key_value", coalesce(col("ContactId"), col("LeadId")))

        # --- Dynamic Prefixing ---
        for c in raw_cols:
            c_lower = c.lower()
            if c_lower not in known_system_keys:
                if c_lower.startswith("a_"):
                    df_norm = df_norm.withColumn(c_lower, col(c).cast("string"))
                else:
                    df_norm = df_norm.withColumn(f"a_{c_lower}", col(c).cast("string"))
        
        # --- THE FIX: Case-Insensitive, Collision-Proof Column Selection ---
        final_cols_set = set() # Used for case-insensitive duplicate checking
        final_cols_list = []   # The actual list passed to Spark
        
        def add_column(col_name):
            if col_name.lower() not in final_cols_set:
                final_cols_list.append(col_name)
                final_cols_set.add(col_name.lower())

        # 1. Force add all base tracking columns first (lowercase)
        base_tracking_cols = [
            'tenant', 'id', 'source_system', 'source_system_object', 
            'source_key_name', 'source_key_value', 'data_timestamp', 
            'created_date', 'created_at', 'updated_at', 'status', 
            'systemmodstamp', 'extraction_timestamp'
        ]
        for c in base_tracking_cols: add_column(c)
        
        # 2. Force add optional parsed core columns if they exist
        optional_core_cols = ['domain', 'name', 'email', 'contact_source_system_object', 'contact_source_key_value', 'campaign_source_key_value', 'event_timestamp']
        for c in optional_core_cols:
            if c in df_norm.columns: add_column(c)
                
        # 3. Add original raw system keys (Like WhoId, WhatId). 
        # NOTE: This safely skips 'Name' or 'Email' because they were already added in step 2!
        for c in raw_cols:
            if c.lower() in known_system_keys:
                add_column(c)
        
        # 4. Add all the 'a_' prefixed custom/standard fields
        for c in df_norm.columns:
            if c.startswith('a_'):
                add_column(c)
        
        df_norm = df_norm.select(*final_cols_list)

    elif source_system == "bigquery":
        df_norm = df_norm \
            .withColumn("tenant", lit(int(customer_id.split('_')[1])).cast(LongType())) \
            .withColumn("source_system", lit("bigquery")) \
            .withColumn("source_system_object", lit("Event")) \
            .withColumn("source_key_name", lit("event_id")) \
            .withColumn("source_key_value", col("id")) \
            .withColumn("id", concat(lit("bigquery_Event_event_id_"), col("id"))) \
            .withColumn("data_timestamp", coalesce(
                to_timestamp(col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
                col("event_timestamp").cast("timestamp")
            )) \
            .withColumn("created_date", coalesce(
                to_timestamp(col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
                col("event_timestamp").cast("timestamp")
            )) \
            .withColumn("created_at", current_timestamp()) \
            .withColumn("updated_at", current_timestamp())

    # --- B. Intra-Batch Dedup ---
    window_spec = Window.partitionBy("source_system", "source_system_object", "source_key_name", "source_key_value").orderBy(col("extraction_timestamp").desc(), col("data_timestamp").desc())
    df_final = df_norm.withColumn("rk", row_number().over(window_spec)).filter(col("rk") == 1).drop("rk", "extraction_timestamp")

    # --- C. Cross-Batch Change Detection ---
    core_attr_cols = ["domain", "name", "email", "contact_source_system_object", "contact_source_key_value", "campaign_source_key_value", "event_timestamp"]
    attr_cols = [c for c in df_final.columns if c.startswith("a_") or c in core_attr_cols]
    
    df_final = df_final.withColumn("record_hash", xxhash64(*[col(c) for c in attr_cols]).cast("string"))

    try:
        target_lookup = active_spark.table(table_full_name).select("id", col("record_hash").alias("t_hash"))
        
        df_final = df_final.join(target_lookup, on="id", how="left") \
                           .filter("t_hash IS NULL OR record_hash != t_hash") \
                           .drop("t_hash")
                           
        if df_final.count() == 0:
            print("Skipping MERGE: No actual data changes detected in this batch.")
            return
    except Exception:
        pass

    # --- D. EXPLICIT SCHEMA EVOLUTION ---
    target_cols_lower = []
    retries = 3
    for attempt in range(retries):
        try:
            target_cols_raw = active_spark.sql(f"SHOW COLUMNS IN {table_full_name}").collect()
            target_cols_lower = [row[0].lower() for row in target_cols_raw]
            break
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2)
            else:
                raise Exception(f"Worker could not find table {table_full_name}. Error: {e}")

    new_cols = []
    for f in df_final.schema:
        if f.name.lower() not in target_cols_lower:
            new_cols.append(f"`{f.name}` {f.dataType.simpleString()}")
            
    if new_cols:
        print(f"✨ Evolving Schema: Adding {len(new_cols)} new columns to {table_full_name}")
        active_spark.sql(f"ALTER TABLE {table_full_name} ADD COLUMNS ({', '.join(new_cols)})")

    # --- E. Dynamic Merge Upsert ---
    df_final.createOrReplaceTempView("batch_updates")
    print(f"Executing MERGE INTO {table_full_name}...")
    
    merge_cols = df_final.columns
    
    update_cols = [c for c in merge_cols if c not in ['tenant', 'id', 'source_system', 'source_system_object', 'source_key_name', 'source_key_value', 'created_at', 'status']]
    
    update_set_parts = []
    for c in update_cols:
        if c in ["updated_at", "data_timestamp", "record_hash"]:
            update_set_parts.append(f"target.`{c}` = source.`{c}`")
        else:
            update_set_parts.append(f"target.`{c}` = COALESCE(source.`{c}`, target.`{c}`)")
            
    update_set_clause = ", ".join(update_set_parts) + ", target.status = 'updated'"
    
    insert_cols = ", ".join([f"`{c}`" for c in merge_cols])
    insert_vals = ", ".join([f"source.`{c}`" if c != 'status' else "'new'" for c in merge_cols])

    active_spark.sql(f"""
        MERGE INTO {table_full_name} AS target
        USING batch_updates AS source
        ON target.id = source.id
        WHEN MATCHED AND source.data_timestamp > COALESCE(target.data_timestamp, CAST('1900-01-01' AS TIMESTAMP)) THEN
            UPDATE SET {update_set_clause}
        WHEN NOT MATCHED THEN
            INSERT ({insert_cols})
            VALUES ({insert_vals})
    """)

# --------------------------------------------------
# 4. Auto Loader Stream Execution
# --------------------------------------------------
@retry_on_failure()
def start_ingestion():
    # Project the file metadata immediately as we read the stream
    df_stream = spark.readStream \
        .format("cloudFiles") \
        .option("cloudFiles.format", "parquet") \
        .option("cloudFiles.inferColumnTypes", "true") \
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
        .option("cloudFiles.schemaLocation", checkpoint_path + "_schema") \
        .load(landing_zone_path) \
        .withColumn("extraction_timestamp", col("_metadata.file_modification_time"))

    query = df_stream.writeStream \
        .foreachBatch(process_bronze_batch) \
        .option("checkpointLocation", checkpoint_path) \
        .trigger(availableNow=True) \
        .start()

    query.awaitTermination()
    
    print("🧹 Running Table Maintenance...")
    spark.sql(f"OPTIMIZE {table_full_name}")
    spark.sql(f"VACUUM {table_full_name} RETAIN 168 HOURS")
    print(f"✅ Bronze Load Complete for {object_name}!")

# Execute the pipeline
start_ingestion()

In [0]:
# from pyspark.sql.functions import col, lit, concat, lower, regexp_replace, split, when, current_timestamp, row_number, coalesce, to_timestamp, last, xxhash64
# from pyspark.sql.window import Window
# from pyspark.sql.types import LongType
# import time

# # --------------------------------------------------
# # 1. Performance, Widget Setup & Retry Decorator
# # --------------------------------------------------
# spark.conf.set("spark.sql.shuffle.partitions", 4)

# dbutils.widgets.text("customer_id", "cust_001")
# dbutils.widgets.text("source_system", "salesforce") 
# dbutils.widgets.text("object_name", "account")      
# dbutils.widgets.text("load_type", "historical")

# customer_id = dbutils.widgets.get("customer_id").lower()
# source_system = dbutils.widgets.get("source_system").lower()
# object_name = dbutils.widgets.get("object_name").lower()
# load_type = dbutils.widgets.get("load_type").lower()

# def retry_on_failure(retries=3, delay=15):
#     def decorator(func):
#         def wrapper(*args, **kwargs):
#             for attempt in range(1, retries + 1):
#                 try:
#                     return func(*args, **kwargs)
#                 except Exception as e:
#                     print(f"Attempt {attempt} failed for {object_name}: {str(e)}")
#                     if attempt == retries: raise e
#                     time.sleep(delay)
#         return wrapper
#     return decorator

# # Define Paths 
# bucket_path = "s3://hgi-databricks-data-lakehouse-dev/"
# landing_zone_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/{load_type}/"
# checkpoint_path = f"{bucket_path}layers/bronze/checkpoints/{load_type}/{source_system}/{customer_id}/{object_name}/"

# # Define Target Table 
# if source_system == "bigquery":
#     target_table_name = "crm_events"
# else:
#     target_table_mapping = {
#         "account": "crm_accounts", 
#         "contact": "crm_contacts",
#         "opportunity": "crm_opportunities", 
#         "campaign": "crm_campaigns", 
#         "user": "crm_users",
#         "event": "crm_events", 
#         "task": "crm_tasks",
#         "campaignmember": "crm_campaign_members"
#     }
#     target_table_name = target_table_mapping.get(object_name, f"crm_{object_name}s")

# catalog_name = "bronze"
# schema_name = customer_id 
# table_full_name = f"{catalog_name}.{schema_name}.{target_table_name}"

# print(f"🚀 Starting Multi-Tenant Bronze Auto Loader: {source_system} -> {object_name} -> {table_full_name} ({load_type})")

# # --------------------------------------------------
# # 2. Minimal Target Table Initialization
# # --------------------------------------------------
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# # Added record_hash to base tracking columns
# spark.sql(f"""
#     CREATE TABLE IF NOT EXISTS {table_full_name} (
#         tenant BIGINT,
#         id STRING,
#         source_system STRING,
#         source_system_object STRING,
#         source_key_name STRING,
#         source_key_value STRING,
#         data_timestamp TIMESTAMP,
#         created_date TIMESTAMP,
#         created_at TIMESTAMP,
#         updated_at TIMESTAMP,
#         status STRING,
#         record_hash STRING
#     )
#     USING DELTA
#     CLUSTER BY (source_system, source_system_object, source_key_name, source_key_value)
# """)

# # --------------------------------------------------
# # 3. Dynamic Normalization & Deduplication Logic
# # --------------------------------------------------
# def process_bronze_batch(batch_df, batch_id):
#     if batch_df.count() == 0:
#         return

#     active_spark = batch_df.sparkSession

#     print(f"Processing Batch {batch_id} with {batch_df.count()} raw records...")
    
#     # Exclude 'extraction_timestamp' from raw_cols so it doesn't trigger dynamic prefixing
#     raw_cols = [c for c in batch_df.columns if c != "extraction_timestamp"]
#     sf_object_name = object_name.capitalize()
    
#     df_norm = batch_df.withColumn("status", lit("processing"))
    
#     # --- A. Client Spec: Schema Normalization & Dynamic Prefixing ---
#     if source_system == "salesforce":
#         df_norm = df_norm \
#             .withColumn("tenant", lit(int(customer_id.split('_')[1])).cast(LongType())) \
#             .withColumn("source_system", lit("salesforce")) \
#             .withColumn("source_system_object", lit(sf_object_name)) \
#             .withColumn("source_key_name", lit("Id")) \
#             .withColumn("source_key_value", col("Id")) \
#             .withColumn("id", concat(lit(f"salesforce_{sf_object_name}_Id_"), col("Id"))) \
#             .withColumn("data_timestamp", coalesce(
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),  
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ssX"),
#                 col("SystemModstamp").cast("timestamp") 
#             )) \
#             .withColumn("systemmodstamp", coalesce(
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),  
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ssX"),
#                 col("SystemModstamp").cast("timestamp") 
#             )) \
#             .withColumn("created_date", coalesce(
#                 to_timestamp(col("CreatedDate"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
#                 to_timestamp(col("CreatedDate"), "yyyy-MM-dd'T'HH:mm:ssX"),
#                 col("CreatedDate").cast("timestamp")
#             )) \
#             .withColumn("created_at", current_timestamp()) \
#             .withColumn("updated_at", current_timestamp())

#         # Base known system keys to exclude from dynamic a_ prefixing
#         known_system_keys = ['id', 'systemmodstamp', 'createddate', 'extraction_timestamp']

#         # --- Object Specific Extraction Logic ---
#         if object_name == "account":
#             known_system_keys.append('name') 
#             df_norm = df_norm.withColumn("name", col("Name")) \
#                              .withColumn("domain", when(col("Website").isNotNull(), lower(regexp_replace(regexp_replace(regexp_replace(col("Website"), r'^https?://', ''), r'^www\.', ''), r'/.*$', ''))).otherwise(col("Website")))
        
#         elif object_name == "contact":
#             known_system_keys.append('email') 
#             df_norm = df_norm.withColumn("email", lower(col("Email"))) \
#                              .withColumn("domain", when(col("Email").contains("@"), split(lower(col("Email")), "@").getItem(1)).otherwise(lit(None)))
        
#         elif object_name in ["task", "event"]:
#             known_system_keys.extend(['whoid', 'whatid', 'activitydate'])
#             df_norm = df_norm.withColumn("contact_source_system_object",
#                 when(col("WhoId").startswith("003"), lit("Contact"))
#                 .when(col("WhoId").startswith("00Q"), lit("Lead"))
#                 .when(col("WhatId").startswith("001"), lit("Account"))
#                 .when(col("WhatId").startswith("006"), lit("Opportunity"))
#                 .when(col("WhatId").startswith("500"), lit("Case"))
#                 .when(col("WhatId").startswith("701"), lit("Campaign"))
#                 .otherwise(lit("Unknown"))
#             ).withColumn("contact_source_key_value", coalesce(col("WhoId"), col("WhatId")))
            
#             if "ActivityDate" in raw_cols:
#                 df_norm = df_norm.withColumn("event_timestamp", col("ActivityDate").cast("timestamp"))

#         elif object_name == "campaignmember":
#             known_system_keys.extend(['campaignid', 'leadid', 'contactid'])
#             df_norm = df_norm.withColumn("campaign_source_key_value", col("CampaignId")) \
#                              .withColumn("contact_source_key_value", coalesce(col("ContactId"), col("LeadId")))

#         # --- Dynamic Prefixing ---
#         for c in raw_cols:
#             c_lower = c.lower()
#             if c_lower not in known_system_keys:
#                 if c_lower.startswith("a_"):
#                     df_norm = df_norm.withColumn(c_lower, col(c).cast("string"))
#                 else:
#                     df_norm = df_norm.withColumn(f"a_{c_lower}", col(c).cast("string"))
        
#         # --- THE FIX: Robust Final Column Selection ---
#         final_cols = [
#             'tenant', 'id', 'source_system', 'source_system_object', 
#             'source_key_name', 'source_key_value', 'data_timestamp', 
#             'created_date', 'created_at', 'updated_at', 'status', 
#             'systemmodstamp', 'extraction_timestamp'
#         ]
        
#         # 1. Append our generated core logic columns
#         optional_core_cols = ['domain', 'name', 'email', 'contact_source_system_object', 'contact_source_key_value', 'campaign_source_key_value', 'event_timestamp']
#         for opt_col in optional_core_cols:
#             if opt_col in df_norm.columns:
#                 final_cols.append(opt_col)
                
#         # 2. THE FIX: Dynamically append ALL known system keys so we never drop raw columns (like WhoId, WhatId)
#         for sys_key in known_system_keys:
#             if sys_key in df_norm.columns and sys_key not in final_cols:
#                 final_cols.append(sys_key)
        
#         # 3. Append all the 'a_' prefixed custom/standard fields
#         final_cols.extend([c for c in df_norm.columns if c.startswith('a_')])
        
#         # Dedup list and apply selection
#         final_cols = list(dict.fromkeys(final_cols))
#         df_norm = df_norm.select(*final_cols)

#     elif source_system == "bigquery":
#         df_norm = df_norm \
#             .withColumn("tenant", lit(int(customer_id.split('_')[1])).cast(LongType())) \
#             .withColumn("source_system", lit("bigquery")) \
#             .withColumn("source_system_object", lit("Event")) \
#             .withColumn("source_key_name", lit("event_id")) \
#             .withColumn("source_key_value", col("id")) \
#             .withColumn("id", concat(lit("bigquery_Event_event_id_"), col("id"))) \
#             .withColumn("data_timestamp", coalesce(
#                 to_timestamp(col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
#                 col("event_timestamp").cast("timestamp")
#             )) \
#             .withColumn("created_date", coalesce(
#                 to_timestamp(col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
#                 col("event_timestamp").cast("timestamp")
#             )) \
#             .withColumn("created_at", current_timestamp()) \
#             .withColumn("updated_at", current_timestamp())

#     # --- B. Intra-Batch Dedup ---
#     window_spec = Window.partitionBy("source_system", "source_system_object", "source_key_name", "source_key_value").orderBy(col("extraction_timestamp").desc(), col("data_timestamp").desc())
#     df_final = df_norm.withColumn("rk", row_number().over(window_spec)).filter(col("rk") == 1).drop("rk", "extraction_timestamp")

#     # --- C. Cross-Batch Change Detection ---
#     core_attr_cols = ["domain", "name", "email", "contact_source_system_object", "contact_source_key_value", "campaign_source_key_value", "event_timestamp"]
#     attr_cols = [c for c in df_final.columns if c.startswith("a_") or c in core_attr_cols]
    
#     df_final = df_final.withColumn("record_hash", xxhash64(*[col(c) for c in attr_cols]).cast("string"))

#     try:
#         target_lookup = active_spark.table(table_full_name).select("id", col("record_hash").alias("t_hash"))
        
#         df_final = df_final.join(target_lookup, on="id", how="left") \
#                            .filter("t_hash IS NULL OR record_hash != t_hash") \
#                            .drop("t_hash")
                           
#         if df_final.count() == 0:
#             print("Skipping MERGE: No actual data changes detected in this batch.")
#             return
#     except Exception:
#         pass

#     # --- D. EXPLICIT SCHEMA EVOLUTION ---
#     target_cols_lower = []
#     retries = 3
#     for attempt in range(retries):
#         try:
#             target_cols_raw = active_spark.sql(f"SHOW COLUMNS IN {table_full_name}").collect()
#             target_cols_lower = [row[0].lower() for row in target_cols_raw]
#             break
#         except Exception as e:
#             if attempt < retries - 1:
#                 time.sleep(2)
#             else:
#                 raise Exception(f"Worker could not find table {table_full_name}. Error: {e}")

#     new_cols = []
#     for f in df_final.schema:
#         if f.name.lower() not in target_cols_lower:
#             new_cols.append(f"`{f.name}` {f.dataType.simpleString()}")
            
#     if new_cols:
#         print(f"✨ Evolving Schema: Adding {len(new_cols)} new columns to {table_full_name}")
#         active_spark.sql(f"ALTER TABLE {table_full_name} ADD COLUMNS ({', '.join(new_cols)})")

#     # --- E. Dynamic Merge Upsert ---
#     df_final.createOrReplaceTempView("batch_updates")
#     print(f"Executing MERGE INTO {table_full_name}...")
    
#     merge_cols = df_final.columns
    
#     update_cols = [c for c in merge_cols if c not in ['tenant', 'id', 'source_system', 'source_system_object', 'source_key_name', 'source_key_value', 'created_at', 'status']]
    
#     update_set_parts = []
#     for c in update_cols:
#         if c in ["updated_at", "data_timestamp"]:
#             update_set_parts.append(f"target.`{c}` = source.`{c}`")
#         else:
#             update_set_parts.append(f"target.`{c}` = COALESCE(source.`{c}`, target.`{c}`)")
            
#     update_set_clause = ", ".join(update_set_parts) + ", target.status = 'updated'"
    
#     insert_cols = ", ".join([f"`{c}`" for c in merge_cols])
#     insert_vals = ", ".join([f"source.`{c}`" if c != 'status' else "'new'" for c in merge_cols])

#     active_spark.sql(f"""
#         MERGE INTO {table_full_name} AS target
#         USING batch_updates AS source
#         ON target.id = source.id
#         WHEN MATCHED AND source.data_timestamp > COALESCE(target.data_timestamp, CAST('1900-01-01' AS TIMESTAMP)) THEN
#             UPDATE SET {update_set_clause}
#         WHEN NOT MATCHED THEN
#             INSERT ({insert_cols})
#             VALUES ({insert_vals})
#     """)

# # --------------------------------------------------
# # 4. Auto Loader Stream Execution
# # --------------------------------------------------
# @retry_on_failure()
# def start_ingestion():
#     # Project the file metadata immediately as we read the stream
#     df_stream = spark.readStream \
#         .format("cloudFiles") \
#         .option("cloudFiles.format", "parquet") \
#         .option("cloudFiles.inferColumnTypes", "true") \
#         .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
#         .option("cloudFiles.schemaLocation", checkpoint_path + "_schema") \
#         .load(landing_zone_path) \
#         .withColumn("extraction_timestamp", col("_metadata.file_modification_time"))

#     query = df_stream.writeStream \
#         .foreachBatch(process_bronze_batch) \
#         .option("checkpointLocation", checkpoint_path) \
#         .trigger(availableNow=True) \
#         .start()

#     query.awaitTermination()
    
#     print("🧹 Running Table Maintenance...")
#     spark.sql(f"OPTIMIZE {table_full_name}")
#     spark.sql(f"VACUUM {table_full_name} RETAIN 168 HOURS")
#     print(f"✅ Bronze Load Complete for {object_name}!")

# # Execute the pipeline
# start_ingestion()

In [0]:
# from pyspark.sql.functions import col, lit, concat, lower, regexp_replace, split, when, current_timestamp, row_number, coalesce, to_timestamp
# from pyspark.sql.window import Window
# from pyspark.sql.types import LongType
# import time

# # --------------------------------------------------
# # 1. Performance, Widget Setup & Retry Decorator
# # --------------------------------------------------
# spark.conf.set("spark.sql.shuffle.partitions", 4)

# dbutils.widgets.text("customer_id", "cust_001")
# dbutils.widgets.text("source_system", "salesforce") 
# dbutils.widgets.text("object_name", "account")      
# dbutils.widgets.text("load_type", "historical")

# customer_id = dbutils.widgets.get("customer_id").lower()
# source_system = dbutils.widgets.get("source_system").lower()
# object_name = dbutils.widgets.get("object_name").lower()
# load_type = dbutils.widgets.get("load_type").lower()

# def retry_on_failure(retries=3, delay=15):
#     def decorator(func):
#         def wrapper(*args, **kwargs):
#             for attempt in range(1, retries + 1):
#                 try:
#                     return func(*args, **kwargs)
#                 except Exception as e:
#                     print(f"Attempt {attempt} failed for {object_name}: {str(e)}")
#                     if attempt == retries: raise e
#                     time.sleep(delay)
#         return wrapper
#     return decorator

# # Define Paths 
# bucket_path = "s3://hgi-databricks-data-lakehouse-dev/"
# landing_zone_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/{load_type}/"
# checkpoint_path = f"{bucket_path}layers_test/bronze/checkpoints/{load_type}/{source_system}/{customer_id}/{object_name}/"

# # Define Target Table 
# if source_system == "bigquery":
#     target_table_name = "crm_events"
# else:
#     target_table_mapping = {
#         "account": "crm_accounts", "contact": "crm_contacts",
#         "opportunity": "crm_opportunities", "campaign": "crm_campaigns", "user": "crm_users"
#     }
#     target_table_name = target_table_mapping.get(object_name, f"crm_{object_name}s")

# catalog_name = "bronze"
# schema_name = customer_id 
# table_full_name = f"{catalog_name}.{schema_name}.{target_table_name}"

# print(f"🚀 Starting Multi-Tenant Bronze Auto Loader: {source_system} -> {object_name} -> {table_full_name} ({load_type})")

# # --------------------------------------------------
# # 2. Minimal Target Table Initialization
# # --------------------------------------------------
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# spark.sql(f"""
#     CREATE TABLE IF NOT EXISTS {table_full_name} (
#         tenant BIGINT,
#         id STRING,
#         source_system STRING,
#         source_system_object STRING,
#         source_key_name STRING,
#         source_key_value STRING,
#         data_timestamp TIMESTAMP,
#         created_date TIMESTAMP,
#         created_at TIMESTAMP,
#         updated_at TIMESTAMP,
#         status STRING 
#     )
#     USING DELTA
#     CLUSTER BY (source_system, source_system_object, source_key_name, source_key_value)
# """)

# # --------------------------------------------------
# # 3. Dynamic Normalization & Deduplication Logic
# # --------------------------------------------------
# def process_bronze_batch(batch_df, batch_id):
#     if batch_df.count() == 0:
#         return

#     active_spark = batch_df.sparkSession

#     print(f"Processing Batch {batch_id} with {batch_df.count()} raw records...")
    
#     # Exclude 'extraction_timestamp' from raw_cols so it doesn't trigger your dynamic 'a_' prefixing
#     raw_cols = [c for c in batch_df.columns if c != "extraction_timestamp"]
#     sf_object_name = object_name.capitalize()
    
#     df_norm = batch_df.withColumn("status", lit("processing"))
    
#     # --- A. Client Spec: Schema Normalization & Dynamic Prefixing ---
#     if source_system == "salesforce":
#         df_norm = df_norm \
#             .withColumn("tenant", lit(int(customer_id.split('_')[1])).cast(LongType())) \
#             .withColumn("source_system", lit("salesforce")) \
#             .withColumn("source_system_object", lit(sf_object_name)) \
#             .withColumn("source_key_name", lit("Id")) \
#             .withColumn("source_key_value", col("Id")) \
#             .withColumn("id", concat(lit(f"salesforce_{sf_object_name}_Id_"), col("Id"))) \
#             .withColumn("data_timestamp", coalesce(
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),  
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ssX"),
#                 col("SystemModstamp").cast("timestamp") 
#             )) \
#             .withColumn("systemmodstamp", coalesce(
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),  
#                 to_timestamp(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ssX"),
#                 col("SystemModstamp").cast("timestamp") 
#             )) \
#             .withColumn("created_date", coalesce(
#                 to_timestamp(col("CreatedDate"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
#                 to_timestamp(col("CreatedDate"), "yyyy-MM-dd'T'HH:mm:ssX"),
#                 col("CreatedDate").cast("timestamp")
#             )) \
#             .withColumn("created_at", current_timestamp()) \
#             .withColumn("updated_at", current_timestamp())

#         # Added extraction_timestamp to known keys as a safeguard
#         known_system_keys = ['id', 'systemmodstamp', 'createddate', 'extraction_timestamp']

#         if object_name == "account":
#             known_system_keys.append('name') 
#             df_norm = df_norm.withColumn("name", col("Name")) \
#                              .withColumn("domain", when(col("Website").isNotNull(), lower(regexp_replace(regexp_replace(regexp_replace(col("Website"), r'https?://', ''), r'www\.', ''), r'/.*$', ''))).otherwise(col("Website")))
        
#         elif object_name == "contact":
#             known_system_keys.append('email') 
#             df_norm = df_norm.withColumn("email", lower(col("Email"))) \
#                              .withColumn("domain", when(col("Email").contains("@"), split(lower(col("Email")), "@").getItem(1)).otherwise(lit(None)))
        
#         # Dynamic Prefixing for all other Salesforce Fields
#         for c in raw_cols:
#             c_lower = c.lower()
#             if c_lower not in known_system_keys:
#                 df_norm = df_norm.withColumn(f"a_{c_lower}", col(c).cast("string"))
        
#         final_cols = ['tenant', 'id', 'source_system', 'source_system_object', 'source_key_name', 'source_key_value', 'data_timestamp', 'created_date', 'created_at', 'updated_at', 'status', 'systemmodstamp']
#         if 'domain' in df_norm.columns: final_cols.append('domain')
#         if 'name' in df_norm.columns: final_cols.append('name')
#         if 'email' in df_norm.columns: final_cols.append('email')
        
#         # Include extraction_timestamp so it passes to the dedup logic
#         final_cols.append("extraction_timestamp")
#         final_cols.extend([c for c in df_norm.columns if c.startswith('a_')])
#         df_norm = df_norm.select(*final_cols)

#     elif source_system == "bigquery":
#         df_norm = df_norm \
#             .withColumn("tenant", lit(int(customer_id.split('_')[1])).cast(LongType())) \
#             .withColumn("source_system", lit("bigquery")) \
#             .withColumn("source_system_object", lit("Event")) \
#             .withColumn("source_key_name", lit("event_id")) \
#             .withColumn("source_key_value", col("id")) \
#             .withColumn("id", concat(lit("bigquery_Event_event_id_"), col("id"))) \
#             .withColumn("data_timestamp", coalesce(
#                 to_timestamp(col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
#                 col("event_timestamp").cast("timestamp")
#             )) \
#             .withColumn("created_date", coalesce(
#                 to_timestamp(col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
#                 col("event_timestamp").cast("timestamp")
#             )) \
#             .withColumn("created_at", current_timestamp()) \
#             .withColumn("updated_at", current_timestamp())

#     # --- B. Intra-Batch Dedup ---
#     # The extraction_timestamp column is already created in the readStream below. No need to extract it here.
#     window_spec = Window.partitionBy("source_system", "source_system_object", "source_key_name", "source_key_value").orderBy(col("extraction_timestamp").desc(), col("data_timestamp").desc())
#     df_final = df_norm.withColumn("rk", row_number().over(window_spec)).filter(col("rk") == 1).drop("rk", "extraction_timestamp")

#     # --- C. EXPLICIT SCHEMA EVOLUTION (The Serverless Sync Fix) ---
#     target_cols_lower = []
#     retries = 3
#     for attempt in range(retries):
#         try:
#             target_cols_raw = active_spark.sql(f"SHOW COLUMNS IN {table_full_name}").collect()
#             target_cols_lower = [row[0].lower() for row in target_cols_raw]
#             break
#         except Exception as e:
#             if attempt < retries - 1:
#                 print(f"Catalog sync delay detected on worker. Retrying schema fetch in 2s... (Attempt {attempt+1}/{retries})")
#                 time.sleep(2)
#             else:
#                 raise Exception(f"Worker could not find table {table_full_name} after {retries} retries. Error: {e}")

#     new_cols = []
#     for f in df_final.schema:
#         if f.name.lower() not in target_cols_lower:
#             new_cols.append(f"`{f.name}` {f.dataType.simpleString()}")
            
#     if new_cols:
#         print(f"✨ Evolving Schema: Adding {len(new_cols)} new columns to {table_full_name}")
#         active_spark.sql(f"ALTER TABLE {table_full_name} ADD COLUMNS ({', '.join(new_cols)})")

#     # --- D. Dynamic Merge Upsert ---
#     df_final.createOrReplaceTempView("batch_updates")
#     print(f"Executing MERGE INTO {table_full_name}...")
    
#     merge_cols = df_final.columns
    
#     update_cols = [c for c in merge_cols if c not in ['tenant', 'id', 'source_system', 'source_system_object', 'source_key_name', 'source_key_value', 'created_at', 'status']]
    
#     update_set_parts = []
#     for c in update_cols:
#         if c in ["updated_at", "data_timestamp"]:
#             update_set_parts.append(f"target.`{c}` = source.`{c}`")
#         else:
#             update_set_parts.append(f"target.`{c}` = COALESCE(source.`{c}`, target.`{c}`)")
            
#     update_set_clause = ", ".join(update_set_parts) + ", target.status = 'updated'"
    
#     insert_cols = ", ".join([f"`{c}`" for c in merge_cols])
#     insert_vals = ", ".join([f"source.`{c}`" if c != 'status' else "'new'" for c in merge_cols])

#     active_spark.sql(f"""
#         MERGE INTO {table_full_name} AS target
#         USING batch_updates AS source
#         ON target.source_system = source.source_system
#            AND target.source_system_object = source.source_system_object
#            AND target.source_key_name = source.source_key_name
#            AND target.source_key_value = source.source_key_value
#         WHEN MATCHED AND source.data_timestamp > COALESCE(target.data_timestamp, CAST('1900-01-01' AS TIMESTAMP)) THEN
#             UPDATE SET {update_set_clause}
#         WHEN NOT MATCHED THEN
#             INSERT ({insert_cols})
#             VALUES ({insert_vals})
#     """)

# # --------------------------------------------------
# # 4. Auto Loader Stream Execution
# # --------------------------------------------------
# @retry_on_failure()
# def start_ingestion():
#     # 💡 FIX: We project the file metadata immediately as we read the stream
#     df_stream = spark.readStream \
#         .format("cloudFiles") \
#         .option("cloudFiles.format", "parquet") \
#         .option("cloudFiles.inferColumnTypes", "true") \
#         .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
#         .option("cloudFiles.schemaLocation", checkpoint_path + "_schema") \
#         .load(landing_zone_path) \
#         .withColumn("extraction_timestamp", col("_metadata.file_modification_time"))

#     query = df_stream.writeStream \
#         .foreachBatch(process_bronze_batch) \
#         .option("checkpointLocation", checkpoint_path) \
#         .trigger(availableNow=True) \
#         .start()

#     query.awaitTermination()
    
#     print("🧹 Running Table Maintenance...")
#     spark.sql(f"OPTIMIZE {table_full_name}")
#     spark.sql(f"VACUUM {table_full_name} RETAIN 168 HOURS")
#     print(f"✅ Bronze Load Complete for {object_name}!")

# # Execute the pipeline
# start_ingestion()